# Bài 2 — Trang 30 — Câu 4 (d): Vẽ biểu đồ phân tán

**Võ Thiện Linh — MSSV: 6451071041**

Sau khi lọc dữ liệu từ ngày 03/01/2018 và tạo `minutes`, vẽ ba scatter plot biểu diễn `co_level`, `humidity`, `temperature` theo thời điểm đo trong ngày.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
columns = ['id', 'device_id', 'co_level', 'humidity', 'temperature', 'time']
candidates = [Path('tb_tracking.xlsx'), Path('LAB1') / 'Phan01' / '6451071041_LAB1_Bai2_Trang30' / 'tb_tracking.xlsx', Path('/content/tb_tracking.xlsx')]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy tb_tracking.xlsx. Thư mục đang chạy: {Path.cwd()}')
raw_df = pd.read_excel(data_path, header=None, names=columns)
data = raw_df.dropna(how='all').copy()
data['id'] = pd.to_numeric(data['id'], errors='coerce')
data['time'] = pd.to_datetime(data['time'], errors='coerce')
data = data.dropna(subset=['id', 'time']).copy()
data = data.loc[data['id'].mod(1).eq(0)].copy()
data = data.drop_duplicates(subset='id', keep='first').copy()
data['id'] = data['id'].astype('int64')
cutoff = pd.Timestamp('2018-01-03')
clean_df = data.loc[data['time'].ge(cutoff)].copy()
clean_df['minutes'] = clean_df['time'].dt.hour * 60 + clean_df['time'].dt.minute
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
plots = [('co_level','Nồng độ khí CO','#e15759'), ('humidity','Độ ẩm','#4e79a7'), ('temperature','Nhiệt độ','#59a14f')]
for ax, (column, label, color) in zip(axes, plots):
    ax.scatter(clean_df['minutes'], clean_df[column], s=12, alpha=0.6, color=color)
    ax.set_title(f'{label} theo thời điểm trong ngày')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel('Thời điểm đo trong ngày (phút từ 00:00)')
axes[-1].set_xlim(0, 1440)
axes[-1].set_xticks(range(0, 1441, 120))
fig.suptitle('Dữ liệu môi trường theo thời điểm đo trong ngày', y=1.01)
fig.tight_layout()
plt.show()
print(f'Số điểm trên mỗi biểu đồ: {len(clean_df)}')
assert len(axes) == 3
assert all(len(ax.collections[0].get_offsets()) == len(clean_df) for ax in axes)